In [2]:
from dotenv import load_dotenv
load_dotenv()
import os

from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    FewShotPromptTemplate,
    MessagesPlaceholder
)
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
from pydantic import BaseModel, Field
from typing import List, Optional

llm = ChatAnthropic(
     api_key=os.getenv("ANTHROPIC_API_KEY"),
    model=os.getenv("LLM_MODEL"),
    base_url=os.getenv("LLM_ENDPOINT")
)
print("Setup complete!")

Setup complete!


## Part 1: Understanding Prompt Templates

LangChain offers different template types for different use cases:

| Template Type | Use Case | Example |
|--------------|----------|--------|
| `PromptTemplate` | Simple text prompts | Summaries, translations |
| `ChatPromptTemplate` | Multi-turn conversations | Chatbots, agents |
| `FewShotPromptTemplate` | Learning from examples | Classification, formatting |

In [4]:
# Simple PromptTemplate
simple_prompt = PromptTemplate.from_template(
    "Summarize the following email in one sentence:\n\n{email}"
)

# Test it
test_email = """
Hi,
I ordered a laptop last week (Order #12345) but haven't received any shipping confirmation.
Can you please check the status? I need it for a presentation next Monday.
Thanks,
John
"""

chain = simple_prompt | llm | StrOutputParser()
result = chain.invoke({"email": test_email})
print("Summary:", result)

Summary: John is requesting a shipping status update on his laptop order (#12345) because he hasn't received confirmation and needs it by Monday for a presentation.


In [5]:
# ChatPromptTemplate with system and human messages
email_classifier_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an email classifier. Classify emails into one of these categories:
- SUPPORT: Technical issues, product problems, complaints
- SALES: Pricing inquiries, purchase requests, quotes
- SPAM: Promotional, unsolicited, suspicious
- GENERAL: Everything else

Respond with ONLY the category name."""),
    ("human", "Classify this email:\n\n{email}")
])

classifier_chain = email_classifier_prompt | llm | StrOutputParser()

# Test with different emails
emails = [
    "My laptop screen is flickering. Order #12345. Please help!",
    "Can you send me a quote for 50 units of Widget Pro?",
    "Congratulations! You've won $1,000,000! Click here to claim!",
    "Just wanted to say thanks for the great service last week."
]

print("Email Classifications:")
print("-" * 40)
for email in emails:
    category = classifier_chain.invoke({"email": email})
    print(f"{category}: {email[:50]}...")

Email Classifications:
----------------------------------------
SUPPORT: My laptop screen is flickering. Order #12345. Plea...
SALES: Can you send me a quote for 50 units of Widget Pro...
SPAM: Congratulations! You've won $1,000,000! Click here...
GENERAL: Just wanted to say thanks for the great service la...


In [6]:
# Define examples for the model to learn from
examples = [
    {
        "email": "My order #5678 arrived damaged. The screen has a crack.",
        "extraction": '{{"order_id": "5678", "issue_type": "damaged_product", "product": "screen", "urgency": "high", "sentiment": "negative"}}'
    },
    {
        "email": "Thanks for the quick delivery! Love the new headphones.",
        "extraction": '{{"order_id": null, "issue_type": "positive_feedback", "product": "headphones", "urgency": "low", "sentiment": "positive"}}'
    },
    {
        "email": "I\'ve been waiting 2 weeks for order #9999. Where is it?!",
        "extraction": '{{"order_id": "9999", "issue_type": "delayed_shipping", "product": null, "urgency": "high", "sentiment": "negative"}}'
    }
]

# Create the example template - use input_variables to specify only the template variables
example_template = PromptTemplate(
    input_variables=["email", "extraction"],
    template="Email: {email}\nExtraction: {extraction}"
)

# Create the few-shot prompt
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_template,
    prefix="Extract structured information from customer emails. Follow the exact JSON format shown in the examples.",
    suffix="Email: {email}\nExtraction:",
    input_variables=["email"]
)

extraction_chain = few_shot_prompt | llm | StrOutputParser()

# Test with a new email
new_email = "Order #1234 never arrived. I need a refund immediately!"
result = extraction_chain.invoke({"email": new_email})
print("Extracted Information:")
print(result)

Extracted Information:
```json
{"order_id": "1234", "issue_type": "missing_package", "product": null, "urgency": "high", "sentiment": "negative"}
```


In [7]:
# Step 1: Classify the email
classify_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify emails as: SUPPORT, SALES, or GENERAL. Respond with only the category."),
    ("human", "{email}")
])

# Step 2: Generate response based on category
respond_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a customer service agent. Generate a response based on the email category.

Category: {category}
Guidelines:
- SUPPORT: Be empathetic, offer to help, ask for details if needed
- SALES: Be professional, provide information, offer to connect with sales team
- GENERAL: Be friendly and helpful"""),
    ("human", "Original email: {email}\n\nGenerate an appropriate response.")
])

# Build the chain
classify_chain = classify_prompt | llm | StrOutputParser()
respond_chain = respond_prompt | llm | StrOutputParser()

def process_email(email: str) -> dict:
    # Step 1: Classify
    category = classify_chain.invoke({"email": email})
    
    # Step 2: Generate response
    response = respond_chain.invoke({
        "email": email,
        "category": category
    })
    
    return {
        "category": category,
        "response": response
    }

# Test
test_email = "I'm interested in purchasing 100 licenses for my company. Can you provide bulk pricing?"
result = process_email(test_email)
print(f"Category: {result['category']}")
print(f"\nGenerated Response:\n{result['response']}")

Category: SALES

Generated Response:
# Response

Subject: Bulk Licensing Inquiry - 100 Licenses

Dear [Customer Name],

Thank you for your interest in purchasing 100 licenses for your company! We're excited about the opportunity to work with you.

I'd be happy to connect you with our sales team, who can provide you with:
- Custom bulk pricing options
- Volume discounts
- Flexible licensing terms tailored to your company's needs
- Implementation support

To get you the most accurate quote and personalized assistance, I recommend reaching out to our sales team. They'll typically respond within one business day.

Would you like me to have a sales representative contact you? If so, please reply with:
- Your preferred contact method (phone/email)
- Best time to reach you
- Any specific requirements for your deployment

Looking forward to helping your company get started!

Best regards,
[Your Name]
Customer Service Team


In [8]:
# Process email in parallel: classify AND extract info at the same time

classify_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify as SUPPORT, SALES, or GENERAL. Reply with only the category."),
    ("human", "{email}")
])

extract_prompt = ChatPromptTemplate.from_messages([
    ("system", """Extract these fields from the email:
- sender_name: (or "Unknown")
- order_id: (or null)
- urgency: high/medium/low
- key_request: one sentence summary

Format as JSON."""),
    ("human", "{email}")
])

sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze sentiment: POSITIVE, NEGATIVE, or NEUTRAL. Reply with only the sentiment."),
    ("human", "{email}")
])

# Create parallel chain
parallel_analysis = RunnableParallel(
    category=classify_prompt | llm | StrOutputParser(),
    extraction=extract_prompt | llm | StrOutputParser(),
    sentiment=sentiment_prompt | llm | StrOutputParser()
)

# Test
test_email = """
Hi, I'm Sarah from ABC Corp.
My order #7890 was supposed to arrive yesterday but tracking shows it's stuck.
I really need this for our product launch on Friday. Can someone help?
Thanks,
Sarah
"""

results = parallel_analysis.invoke({"email": test_email})

print("=" * 60)
print("PARALLEL EMAIL ANALYSIS")
print("=" * 60)
print(f"\nCategory: {results['category']}")
print(f"\nSentiment: {results['sentiment']}")
print(f"\nExtracted Info:\n{results['extraction']}")

PARALLEL EMAIL ANALYSIS

Category: SUPPORT

Sentiment: NEGATIVE

Extracted Info:
```json
{
  "sender_name": "Sarah",
  "order_id": "#7890",
  "urgency": "high",
  "key_request": "Investigate delayed order #7890 that is needed for product launch on Friday"
}
```


In [9]:
# Different response templates based on category

support_response_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a technical support specialist. Be empathetic and solution-oriented.
Always:
1. Acknowledge the issue
2. Ask clarifying questions if needed
3. Provide immediate steps they can try
4. Offer to escalate if needed"""),
    ("human", "Respond to this support email:\n{email}")
])

sales_response_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a sales representative. Be professional and informative.
Always:
1. Thank them for their interest
2. Provide relevant information
3. Offer to schedule a call
4. Include a clear next step"""),
    ("human", "Respond to this sales inquiry:\n{email}")
])

general_response_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly customer service agent. Be helpful and warm."),
    ("human", "Respond to this email:\n{email}")
])

# Create response chains
support_chain = support_response_prompt | llm | StrOutputParser()
sales_chain = sales_response_prompt | llm | StrOutputParser()
general_chain = general_response_prompt | llm | StrOutputParser()

def route_email(email: str) -> str:
    """Classify email and route to appropriate response chain."""
    # First classify
    category = classify_chain.invoke({"email": email}).strip().upper()
    
    # Route to appropriate chain
    if "SUPPORT" in category:
        response = support_chain.invoke({"email": email})
        dept = "Technical Support"
    elif "SALES" in category:
        response = sales_chain.invoke({"email": email})
        dept = "Sales Team"
    else:
        response = general_chain.invoke({"email": email})
        dept = "Customer Service"
    
    return f"[Routed to: {dept}]\n\n{response}"

# Test with different emails
print("=" * 60)
print("SUPPORT EMAIL")
print("=" * 60)
print(route_email("My software keeps crashing when I try to export. Error code: E-500"))

print("\n" + "=" * 60)
print("SALES EMAIL")
print("=" * 60)
print(route_email("We're evaluating your enterprise plan for our team of 50. What's included?"))

SUPPORT EMAIL
[Routed to: Technical Support]

# Support Response

Subject: Re: Export Crashing Issue - Error E-500

---

Hi there,

Thank you for reaching out, and I'm sorry you're experiencing crashes when exporting. That's definitely frustrating, and I'm here to help get this resolved for you.

**I've found that Error E-500 often relates to file size or format compatibility.** Let me ask a couple of quick questions so I can narrow this down:

1. **What file format are you exporting to?** (PDF, Excel, CSV, etc.)
2. **Approximately how large is the file** you're trying to export?
3. **Did this recently start happening**, or is it the first time you've tried exporting?

---

**In the meantime, here are some immediate steps to try:**

1. **Restart the software** – Close completely and reopen
2. **Try exporting a smaller dataset** – This helps identify if it's a size issue
3. **Check available disk space** – Ensure you have at least 1-2 GB free
4. **Clear temporary files** – Often resolve

In [10]:
# Define a Pydantic model for structured output
class EmailAnalysis(BaseModel):
    """Structured analysis of a customer email."""
    category: str = Field(description="SUPPORT, SALES, or GENERAL")
    urgency: str = Field(description="high, medium, or low")
    sentiment: str = Field(description="positive, negative, or neutral")
    order_id: Optional[str] = Field(description="Order ID if mentioned, otherwise null")
    customer_name: Optional[str] = Field(description="Customer name if provided")
    summary: str = Field(description="One sentence summary of the email")
    suggested_action: str = Field(description="Recommended next step")

# Create parser with the Pydantic model
json_parser = JsonOutputParser(pydantic_object=EmailAnalysis)

# Create prompt with format instructions
structured_prompt = ChatPromptTemplate.from_messages([
    ("system", """Analyze customer emails and extract structured information.

{format_instructions}

Be accurate and concise."""),
    ("human", "Analyze this email:\n\n{email}")
]).partial(format_instructions=json_parser.get_format_instructions())

# Build chain
structured_chain = structured_prompt | llm | json_parser

# Test
test_email = """
Subject: URGENT - Order #5555 Missing Items

Hi,
I'm Mike Thompson and I just received my order #5555 but 2 items are missing!
I ordered 5 widgets but only received 3. I need the other 2 ASAP for a client delivery tomorrow.
Please fix this immediately.
Mike
"""

analysis = structured_chain.invoke({"email": test_email})

print("=" * 60)
print("STRUCTURED EMAIL ANALYSIS")
print("=" * 60)
for key, value in analysis.items():
    print(f"{key}: {value}")

STRUCTURED EMAIL ANALYSIS
category: SUPPORT
urgency: high
sentiment: negative
order_id: 5555
customer_name: Mike Thompson
summary: Customer received order #5555 with 2 missing widgets out of 5 ordered and needs them urgently for a client delivery tomorrow.
suggested_action: Immediately verify inventory, confirm missing items, and expedite shipment of the 2 widgets or provide alternative resolution within 24 hours.


In [11]:
class EmailProcessor:
    """Complete email processing system with classification, extraction, and response generation."""
    
    def __init__(self, llm):
        self.llm = llm
        self._setup_chains()
    
    def _setup_chains(self):
        # Analysis chain (parallel)
        self.analysis_chain = RunnableParallel(
            category=ChatPromptTemplate.from_messages([
                ("system", "Classify as SUPPORT, SALES, SPAM, or GENERAL. Reply with only the category."),
                ("human", "{email}")
            ]) | self.llm | StrOutputParser(),
            
            urgency=ChatPromptTemplate.from_messages([
                ("system", "Rate urgency as HIGH, MEDIUM, or LOW. Reply with only the rating."),
                ("human", "{email}")
            ]) | self.llm | StrOutputParser(),
            
            sentiment=ChatPromptTemplate.from_messages([
                ("system", "Rate sentiment as POSITIVE, NEGATIVE, or NEUTRAL. Reply with only the rating."),
                ("human", "{email}")
            ]) | self.llm | StrOutputParser()
        )
        
        # Response templates
        self.response_templates = {
            "SUPPORT": ChatPromptTemplate.from_messages([
                ("system", """You are a support specialist. Write a helpful, empathetic response.
Urgency level: {urgency}
Customer sentiment: {sentiment}
Adapt your tone accordingly."""),
                ("human", "Original email: {email}\n\nWrite a response.")
            ]),
            "SALES": ChatPromptTemplate.from_messages([
                ("system", "You are a sales representative. Write a professional, informative response."),
                ("human", "Original email: {email}\n\nWrite a response.")
            ]),
            "GENERAL": ChatPromptTemplate.from_messages([
                ("system", "You are a customer service agent. Write a friendly, helpful response."),
                ("human", "Original email: {email}\n\nWrite a response.")
            ]),
            "SPAM": None  # No response for spam
        }
    
    def process(self, email: str) -> dict:
        """Process an email through the complete pipeline."""
        # Step 1: Analyze in parallel
        analysis = self.analysis_chain.invoke({"email": email})
        category = analysis["category"].strip().upper()
        
        # Step 2: Route and generate response (if not spam)
        if "SPAM" in category:
            response = "[SPAM DETECTED - No response generated]"
            routing = "Spam Folder"
        else:
            # Determine routing
            if "SUPPORT" in category:
                routing = "Technical Support Queue"
                template = self.response_templates["SUPPORT"]
            elif "SALES" in category:
                routing = "Sales Team"
                template = self.response_templates["SALES"]
            else:
                routing = "General Inbox"
                template = self.response_templates["GENERAL"]
            
            # Generate response
            response_chain = template | self.llm | StrOutputParser()
            response = response_chain.invoke({
                "email": email,
                "urgency": analysis["urgency"],
                "sentiment": analysis["sentiment"]
            })
        
        return {
            "category": category,
            "urgency": analysis["urgency"].strip(),
            "sentiment": analysis["sentiment"].strip(),
            "routing": routing,
            "response": response
        }

# Create processor
processor = EmailProcessor(llm)
print("Email processor ready!")

Email processor ready!


In [12]:
# Test the complete pipeline
test_emails = [
    {
        "subject": "Support Request",
        "body": """Hi, my account has been locked for 3 days and I can't access my files.
This is affecting my work. Please help urgently! - David"""
    },
    {
        "subject": "Enterprise Inquiry",
        "body": """Hello, our company is interested in your enterprise solution for 200+ users.
Could you provide pricing and a demo? Best regards, Jennifer (CTO, TechCorp)"""
    },
    {
        "subject": "Free Money!!!",
        "body": """YOU HAVE WON $10,000,000!!! Click here NOW to claim your prize!!!
Limited time offer! Send your bank details immediately!"""
    }
]

for email in test_emails:
    print("\n" + "=" * 60)
    print(f"PROCESSING: {email['subject']}")
    print("=" * 60)
    
    result = processor.process(email["body"])
    
    print(f"\n Analysis:")
    print(f"   Category: {result['category']}")
    print(f"   Urgency: {result['urgency']}")
    print(f"   Sentiment: {result['sentiment']}")
    print(f"   Routing: {result['routing']}")
    print(f"\n Generated Response:")
    print(f"   {result['response'][:300]}..." if len(result['response']) > 300 else f"   {result['response']}")


PROCESSING: Support Request

 Analysis:
   Category: SUPPORT
   Urgency: HIGH
   Sentiment: NEGATIVE
   Routing: Technical Support Queue

 Generated Response:
   # Response to David

Subject: Urgent: Account Access Restored - Here's What Happened

Hi David,

Thank you for reaching out, and I sincerely apologize for the frustration this has caused over the past three days. I understand how critical this is when it's affecting your work—that's our priority too...

PROCESSING: Enterprise Inquiry

 Analysis:
   Category: SALES
   Urgency: # HIGH
   Sentiment: NEUTRAL
   Routing: Sales Team

 Generated Response:
   **Subject: Re: Enterprise Solution Inquiry – Pricing & Demo**

---

Dear Jennifer,

Thank you for reaching out, and I'm pleased to hear that TechCorp is interested in our enterprise solution. I'd be happy to assist you with both pricing information and a demonstration.

**Next Steps:**

To provide y...

PROCESSING: Free Money!!!

 Analysis:
   Category: SPAM
   Urgency: LOW

In [16]:
detect_prompt = ChatPromptTemplate.from_messages([
    ("system", "Detect the language of the text. Reply with only the language name."),
    ("human", "{text}")
])


translate_prompt = ChatPromptTemplate.from_messages([
    ("system", "Translate the following {source_language} text to English. If already English, return as-is."),
    ("human", "{text}")
])

In [17]:
# Step 3: Summary prompt
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the following text in one concise sentence."),
    ("human", "{text}")
])
 
# Create chains
detect_chain = detect_prompt | llm | StrOutputParser()
translate_chain = translate_prompt | llm | StrOutputParser()
summary_chain = summary_prompt | llm | StrOutputParser()
 
# Build the complete chain
def translation_chain(text: str):
    # Detect language
    language = detect_chain.invoke({"text": text}).strip()
 
    # Translate
    translated_text = translate_chain.invoke({
        "text": text,
        "source_language": language
    })
 
    # Summarize
    summary = summary_chain.invoke({
        "text": translated_text
    })
 
    return {
        "language": language,
        "translated_text": translated_text,
        "summary": summary
    }
 
# Test
test_text = "Bonjour, je voudrais réserver une table pour deux personnes ce soir à 20 heures."
 
result = translation_chain(test_text)
 
print("=" * 50)
print("Translation Chain Result")
print("=" * 50)
print(f"Detected Language : {result['language']}")
print(f"\nEnglish Translation:\n{result['translated_text']}")
print(f"\nSummary:\n{result['summary']}")
 

 

Translation Chain Result
Detected Language : French

English Translation:
Hello, I would like to reserve a table for two people this evening at 8 PM.

Summary:
I appreciate your interest, but I'm an AI assistant without access to restaurant reservation systems or real-time booking information. 

To make a reservation, I'd recommend:

1. **Call the restaurant directly** - this is usually the fastest way
2. **Use online platforms** like OpenTable, Resy, or Yelp
3. **Check the restaurant's website** for their booking system
4. **Visit in person** if it's nearby

Which restaurant are you trying to book? I'd be happy to help you find their contact information or suggest alternative options!


In [18]:
sample_review = """
I bought this laptop 3 months ago. The screen quality is amazing and the battery lasts all day.
However, the keyboard feels a bit mushy and it runs hot when gaming. For the price, it's decent
but not exceptional. Would recommend for casual users but not power users.
"""
 
# Sentiment prompt
sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "Analyze the overall sentiment of the review. Reply with only POSITIVE, NEGATIVE, or NEUTRAL."),
    ("human", "{review}")
])
 
# Pros prompt
pros_prompt = ChatPromptTemplate.from_messages([
    ("system", "List the key pros mentioned in the review."),
    ("human", "{review}")
])
 
# Cons prompt
cons_prompt = ChatPromptTemplate.from_messages([
    ("system", "List the key cons mentioned in the review."),
    ("human", "{review}")
])
 
# Rating prompt
rating_prompt = ChatPromptTemplate.from_messages([
    ("system", "Predict the star rating (1-5). Reply with only the number."),
    ("human", "{review}")
])
 
# Create parallel chain
review_analyzer = RunnableParallel(
    sentiment=sentiment_prompt | llm | StrOutputParser(),
    pros=pros_prompt | llm | StrOutputParser(),
    cons=cons_prompt | llm | StrOutputParser(),
    rating=rating_prompt | llm | StrOutputParser()
)
 
# Test with the sample review
result = review_analyzer.invoke({"review": sample_review})
 
print("=" * 50)
print("PRODUCT REVIEW ANALYSIS")
print("=" * 50)
print(f"Sentiment : {result['sentiment']}")
print(f"\nPros:\n{result['pros']}")
print(f"\nCons:\n{result['cons']}")
print(f"\nPredicted Rating : {result['rating']}")

PRODUCT REVIEW ANALYSIS
Sentiment : NEUTRAL

Pros:
# Key Pros Mentioned in the Review:

1. **Amazing screen quality** - The display is highlighted as a standout feature
2. **All-day battery life** - The battery performance is strong and long-lasting
3. **Decent value for the price** - Offers reasonable value for casual users

Cons:
# Key Cons Mentioned in the Review

1. **Mushy keyboard feel** - The keyboard lacks responsiveness/tactile feedback
2. **Runs hot during gaming** - Thermal management issues when under load
3. **Not exceptional for the price** - Value proposition is mediocre
4. **Not suitable for power users** - Limited capability for demanding tasks

Predicted Rating : 3


In [20]:
sample_document = """
SERVICE AGREEMENT
...
"""

In [21]:
# Extract dates and parties
extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract all key dates and parties involved from the legal document."),
    ("human", "{document}")
])
 
# Identify document type
type_prompt = ChatPromptTemplate.from_messages([
    ("system", "Identify the document type (Contract, Agreement, Notice, NDA, etc.). Reply with only the document type."),
    ("human", "{document}")
])
 
# Summarize obligations
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the main obligations of the parties in 3-5 bullet points."),
    ("human", "{document}")
])
 
# Flag risks
risk_prompt = ChatPromptTemplate.from_messages([
    ("system", "Identify potential legal risks or concerns mentioned in the document."),
    ("human", "{document}")
])
 
# Build parallel pipeline
document_pipeline = RunnableParallel(
    extraction=extract_prompt | llm | StrOutputParser(),
    document_type=type_prompt | llm | StrOutputParser(),
    obligations=summary_prompt | llm | StrOutputParser(),
    risks=risk_prompt | llm | StrOutputParser()
)
 
# Run the pipeline
result = document_pipeline.invoke({"document": sample_document})
 
print("=" * 60)
print("LEGAL DOCUMENT ANALYSIS")
print("=" * 60)
 
print(f"\nDocument Type:\n{result['document_type']}")
print(f"\nDates & Parties:\n{result['extraction']}")
print(f"\nMain Obligations:\n{result['obligations']}")
print(f"\nPotential Risks:\n{result['risks']}")

LEGAL DOCUMENT ANALYSIS

Document Type:
Service Agreement

Dates & Parties:
I'd be happy to help you extract key dates and parties from a service agreement, but I don't see the actual document content in your message. You've included the header "SERVICE AGREEMENT" followed by "..." but the full document text is missing.

Please share the complete service agreement, and I'll extract:

1. **Key Dates** (such as):
   - Effective date
   - Expiration/termination date
   - Renewal dates
   - Milestone dates
   - Payment due dates

2. **Parties Involved** (such as):
   - Service provider(s)
   - Client(s)/customer(s)
   - Authorized representatives
   - Signatories

Please paste the full document text, and I'll provide you with a clear, organized summary of all key dates and parties.

Main Obligations:
# Service Agreement Summary

I'd be happy to summarize the main obligations, but I don't see the full Service Agreement text in your message. You've only included the header "SERVICE AGREEMENT